### Middleware

In [11]:
import os

from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

### Summrization Middleware

In [12]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

from langgraph.checkpoint.memory import InMemorySaver

from langchain_core.messages import HumanMessage,SystemMessage

from langchain_core.tools import tool

model=init_chat_model("groq:openai/gpt-oss-20b")

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A369429450>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A369429E50>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [13]:
#Message Length Based Summarization

agent=create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10), #summarize after 10 messages
            keep=("messages",4)
        )
    ]
)

#run with thread id
config={
    "configurable":{
        "thread_id":"test-1"
    }
}

#test data

questions=[
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for question in questions:
    response=agent.invoke({
        "messages":[
            HumanMessage(content=question)
        ]
    },config)

    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='c82b430f-2f31-4824-9373-4a19d2a7ef0b'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'The user asks: "What is 2+2?" It\'s a straightforward question. The answer is 4. Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 78, 'total_tokens': 124, 'completion_time': 0.054831361, 'completion_tokens_details': {'reasoning_tokens': 27}, 'prompt_time': 0.006498152, 'prompt_tokens_details': None, 'queue_time': 0.466764569, 'total_time': 0.061329513}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_816fc87d18', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a17962-c1c5-75d1-8b84-96b40a442633-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 46, 'total_tokens': 124, 'output_toke

In [14]:
#Token Size Summarization
@tool
def search_hostels(city:str)->str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent=create_agent(
    model=model,
    tools=[search_hostels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)

config={
    "configurable":{
        "thread_id":"test-1"
    }
}

def count_tokens(messages):
    total_chars=sum(len(str(message.content)) for message in messages)

    return total_chars//4

#Run test
cities=["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response=agent.invoke(
        {
            "messages": [
                HumanMessage(content=f"Find hotels in {city}")
            ]
        },
        config=config
    )
    
    tokens=count_tokens(response["messages"])

    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    
    print(f"{(response['messages'])}")

Paris: ~210 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='4b90cd1e-b434-46c5-b87a-523ab339348e'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the function search_hostels with city="Paris".', 'tool_calls': [{'id': 'fc_5227b962-9a63-4b80-8249-4978ed5dc954', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hostels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 129, 'total_tokens': 168, 'completion_time': 0.043389952, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.006140144, 'prompt_tokens_details': None, 'queue_time': 0.320191941, 'total_time': 0.049530096}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d3e146e1a5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a17962-d48e-7401-bfed-72eb42570a6a-0', tool_call

### Human In The Loop Middleware

In [15]:
#used for pausing agent for human approval regarding acceptance,editing,rejection of tool calls before they execute

from langchain.agents.middleware import HumanInTheLoopMiddleware

def read_email_tool(email:str)->str:
    """Mock Function to read an email by its id"""
    return f"Email Content for ID: {email}"

def send_email_tool(recipient:str,subject:str,body:str)->str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject: {subject}"

agent=create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "send_email_tool":{
                "allowed_decisions":["approve","edit","reject"]
            },
            "read_email_tool":False,
        }
    )]
)


config={"configurable":{"thread_id":"test-approve"}}

email="varad0404@gmail.com"
subject="greetings"
body="how are you"

result=agent.invoke(
    {
        "messages": [
            HumanMessage(content=f"send email to {email}, with subject: {subject} containing text: {body}")
        ]
    },
    config=config
)

result

{'messages': [HumanMessage(content='send email to varad0404@gmail.com, with subject: greetings containing text: how are you', additional_kwargs={}, response_metadata={}, id='4762da39-df4e-4f4f-9797-513f4b79cdbb'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the send_email_tool. The user wants to send an email to varad0404@gmail.com with subject "greetings" containing text: "how are you". So we call send_email_tool with body "how are you", recipient "varad0404@gmail.com", subject "greetings".', 'tool_calls': [{'id': 'fc_36d5117a-a7c8-4c2e-9d76-3f0fc7348ae7', 'function': {'arguments': '{"body":"how are you","recipient":"varad0404@gmail.com","subject":"greetings"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 175, 'total_tokens': 280, 'completion_time': 0.144646592, 'completion_tokens_details': {'reasoning_tokens': 65}, 'prompt_time': 0.008527945, 'prompt_tokens_details

In [16]:
#approve

from langgraph.types import Command

if "__interrupt__" in result:
    print("PAUSED waiting for approval...")

    result=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result['messages'][-1].content}")

PAUSED waiting for approval...
Result: ✅ Your email has been sent to **varad0404@gmail.com** with the subject **“greetings”** and the message **“how are you.”**


In [20]:
# reject
agent=create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "send_email_tool":{
                "allowed_decisions":["approve","edit","reject"]
            },
            "read_email_tool":False,
        }
    )]
)


config={"configurable":{"thread_id":"test-approve"}}

email="varad0404@gmail.com"
subject="greetings"
body="how are you"

result=agent.invoke(
    {
        "messages": [
            HumanMessage(content=f"send email to {email}, with subject: {subject} containing text: {body}")
        ]
    },
    config=config
)

if "__interrupt__" in result:
    print("PAUSED waiting for approval...")

    result=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"reject"}
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result['messages'][-1].content}")

PAUSED waiting for approval...
Result: I’m sorry, but I can’t send the email directly right now.  
Would you like me to draft the email for you so you can copy it into your own mail client?


In [22]:
#edit
agent=create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "send_email_tool":{
                "allowed_decisions":["approve","edit","reject"]
            },
            "read_email_tool":False,
        }
    )]
)


config={"configurable":{"thread_id":"test-approve"}}

correct_email="varad0404@gmail.com"
wrong_email="vrad0404@gmail.com"
subject="greetings"
body="how are you"

result=agent.invoke(
    {
        "messages": [
            HumanMessage(content=f"send email to {email}, with subject: {subject} containing text: {body}")
        ]
    },
    config=config
)

if "__interrupt__" in result:
    print("PAUSED waiting for approval...")

    result=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {
                        "type":"edit",
                        "edited_action":{
                            "name":"send_email_tool",
                            "args":{
                                "recipient": correct_email,
                                "subject": subject,
                                "body": body
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result['messages'][-1].content}")

PAUSED waiting for approval...
Result: ✅ Email sent to varad0404@gmail.com with subject “greetings” and body “how are you”.
